# Visual Style Learning: Kaggle GPU Run

Run from top to bottom. This notebook uses real Ollama Vision and real sequential DeepSeek, Qwen, and Llama agents; it never falls back to mock. Enable Internet and a T4x2-or-better GPU in Kaggle settings first.

The first run is intentionally small: 3 videos, 8 frames/video, 1 cycle. YouTube cookies belong in Kaggle Secrets as `YOUTUBE_COOKIES`, never in cells or Git.

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/rzyasine1-sketch/AI.git"
GIT_BRANCH = "main"
CHANNEL_URL = ""  # Set this to the channel to analyze.
TARGET_VIDEOS = 3
FRAMES_PER_VIDEO = 8
CYCLES = 1
INVENTORY_LIMIT = 500
MIN_DATASET_SAMPLES = 100
MIN_STYLE_CONFIDENCE = 0.75
LORA_EPOCHS = 1
BASE_IMAGE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MIN_GPU_VRAM_GIB = 10
MIN_FREE_DISK_GIB = 20
PROJECT_DIR = Path("/kaggle/working/AI")
WORK_DIR = Path("/kaggle/working")

assert CHANNEL_URL.startswith(("https://www.youtube.com/", "https://youtube.com/")), "BLOCKED: set CHANNEL_URL in this cell to a valid YouTube channel URL."
print({"target_videos": TARGET_VIDEOS, "frames_per_video": FRAMES_PER_VIDEO, "cycles": CYCLES, "resume": True})

In [ ]:
import json
import shutil
import subprocess

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    check=False, capture_output=True, text=True,
)
if gpu_query.returncode != 0 or not gpu_query.stdout.strip():
    raise RuntimeError(f"BLOCKED: no accessible NVIDIA GPU. {gpu_query.stderr.strip()}")
gpus = []
for line in gpu_query.stdout.strip().splitlines():
    name, memory_mib = line.rsplit(",", 1)
    gpus.append({"name": name.strip(), "vram_gib": int(memory_mib.strip()) / 1024})
free_disk_gib = shutil.disk_usage(WORK_DIR).free / 1024**3
print(json.dumps({"gpus": gpus, "free_disk_gib": round(free_disk_gib, 2)}, indent=2))
if max(gpu["vram_gib"] for gpu in gpus) < MIN_GPU_VRAM_GIB:
    raise RuntimeError(f"BLOCKED: each usable GPU must provide at least {MIN_GPU_VRAM_GIB} GiB VRAM.")
if free_disk_gib < MIN_FREE_DISK_GIB:
    raise RuntimeError(f"BLOCKED: need at least {MIN_FREE_DISK_GIB} GiB free disk for models, frames, and checkpoints.")
max_vram_gib = max(gpu["vram_gib"] for gpu in gpus)
VISION_MODEL = "qwen3-vl:4b" if max_vram_gib >= 24 else "qwen3-vl:2b"
AGENT_MODELS = {"deepseek": "deepseek-r1:1.5b", "qwen": "qwen2.5:0.5b", "llama": "llama3.2:1b"}
print({"selected_vision_model": VISION_MODEL, "selection_reason": f"largest single GPU has {max_vram_gib:.1f} GiB VRAM"})

In [ ]:
import os

clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(PROJECT_DIR)],
    check=False, capture_output=True, text=True,
)
if clone.returncode != 0:
    raise RuntimeError(f"BLOCKED: could not clone project source: {clone.stderr.strip()}")
os.chdir(PROJECT_DIR)
required_files = ["main.py", "run.py", "pipeline/real_channel.py", "vision/local_backend.py", "setup.sh"]
missing = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if missing:
    raise RuntimeError("BLOCKED: cloned Git branch does not contain the current project source: " + ", ".join(missing) + ". Publish the current Codespace source to this branch, then rerun.")
print("Project source verified:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
setup_env = os.environ.copy()
setup_env["INSTALL_OLLAMA"] = "1"
subprocess.run(["bash", "setup.sh"], cwd=PROJECT_DIR, env=setup_env, check=True)
print("Python dependencies and Ollama runtime installation completed.")

In [ ]:
import time
from urllib.request import urlopen

if shutil.which("ollama") is None:
    raise RuntimeError("BLOCKED: Ollama executable is unavailable after setup.")
try:
    with urlopen("http://127.0.0.1:11434/api/version", timeout=2) as response:
        print("Existing Ollama runtime:", response.read().decode())
except OSError:
    with (WORK_DIR / "ollama.log").open("a", encoding="utf-8") as ollama_log:
        ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for attempt in range(60):
        try:
            with urlopen("http://127.0.0.1:11434/api/version", timeout=2) as response:
                print("Ollama runtime:", response.read().decode())
                break
        except OSError:
            time.sleep(2)
    else:
        raise RuntimeError("BLOCKED: Ollama did not become ready; inspect /kaggle/working/ollama.log.")

In [ ]:
import yaml

model_order = [VISION_MODEL, AGENT_MODELS["deepseek"], AGENT_MODELS["qwen"], AGENT_MODELS["llama"]]
for model in model_order:
    print(f"Pulling required model sequentially: {model}")
    subprocess.run(["ollama", "pull", model], check=True)

config_path = PROJECT_DIR / "config.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
config.setdefault("vision", {}).update({"backend": "ollama", "model_name": VISION_MODEL, "base_url": "http://127.0.0.1:11434", "timeout": 600, "keep_alive": "5m"})
for name, model in AGENT_MODELS.items():
    config.setdefault("models", {}).setdefault(name, {}).update({"name": model, "backend": "ollama"})
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print("Configured Vision and sequential agent models:", {"vision": VISION_MODEL, **AGENT_MODELS})

In [ ]:
from models.ollama_backend import OllamaBackend
from vision.local_backend import LocalVisionBackend

subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)
vision_backend = LocalVisionBackend(config["vision"])
if not vision_backend.is_available():
    raise RuntimeError("BLOCKED: Vision model unavailable or not vision-capable: " + vision_backend.availability_error)
agent_checks = {}
for name, model in AGENT_MODELS.items():
    backend = OllamaBackend(name=name, config={"model_name": model, "base_url": "http://127.0.0.1:11434", "keep_alive": 0})
    agent_checks[name] = backend.is_available()
if not all(agent_checks.values()):
    raise RuntimeError(f"BLOCKED: required real agent model unavailable: {agent_checks}")
print({"vision_model_available": True, "vision_capability_verified": True, "agents_available": agent_checks, "mock_used": False})

In [ ]:
import os
import tempfile

cookies_path = None
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
    try:
        cookie_text = secrets_client.get_secret("YOUTUBE_COOKIES")
    except Exception:  # noqa: BLE001
        cookie_text = None
    if cookie_text:
        fd, cookies_path = tempfile.mkstemp(prefix="yt_cookies_", suffix=".txt", dir="/tmp")
        os.fchmod(fd, 0o600)
        with os.fdopen(fd, "w", encoding="utf-8") as cookie_file:
            cookie_file.write(cookie_text)
        del cookie_text
    try:
        hf_token = secrets_client.get_secret("HF_TOKEN")
    except Exception:  # noqa: BLE001
        hf_token = None
except ImportError:
    print("Kaggle Secrets API unavailable; proceeding without optional cookies/HF token.")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print({"youtube_cookies_secret_available": cookies_path is not None, "hf_token_available": bool(hf_token)})

In [ ]:
command = [
    sys.executable, "run.py", "--overnight", "--channel", CHANNEL_URL,
    "--target-videos", str(TARGET_VIDEOS), "--frames-per-video", str(FRAMES_PER_VIDEO),
    "--cycles", str(CYCLES), "--inventory-limit", str(INVENTORY_LIMIT),
    "--min-dataset-samples", str(MIN_DATASET_SAMPLES),
    "--min-style-confidence", str(MIN_STYLE_CONFIDENCE),
    "--lora-epochs", str(LORA_EPOCHS), "--base-image-model", BASE_IMAGE_MODEL,
]
if cookies_path:
    command.extend(["--cookies", cookies_path])
pipeline_env = os.environ.copy()
try:
    result = subprocess.run(command, cwd=PROJECT_DIR, env=pipeline_env, check=False, capture_output=True, text=True)
finally:
    if cookies_path and Path(cookies_path).exists():
        Path(cookies_path).unlink()
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    raise RuntimeError("BLOCKED: overnight real pipeline did not complete successfully. Check data/real_run_report.json.")

In [ ]:
import cv2

report_path = PROJECT_DIR / "data/real_run_report.json"
if not report_path.is_file():
    raise RuntimeError("BLOCKED: real run report is missing.")
report = json.loads(report_path.read_text(encoding="utf-8"))
if report.get("status") != "complete":
    raise RuntimeError(f"BLOCKED: real pipeline status is {report.get('status')}: {report.get('blocked_by')}")
if report.get("vision_status") not in (None, "available"):
    raise RuntimeError(f"BLOCKED: Vision model status was {report['vision_status']}")
dataset_path = PROJECT_DIR / report["dataset_path"]
records = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]
if not records:
    raise RuntimeError("BLOCKED: real visual dataset is empty.")
for record in records:
    image_path = PROJECT_DIR / "data" / record["image"]
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None or image.size == 0 or image_path.stat().st_size == 0:
        raise RuntimeError(f"BLOCKED: invalid dataset image {image_path}")
    if not record.get("raw_observation") or not record.get("visual_analysis"):
        raise RuntimeError(f"BLOCKED: missing raw/visual analysis for {image_path}")
    if not str(record.get("analysis_backend", "")).startswith("ollama-vision:"):
        raise RuntimeError(f"BLOCKED: frame was not analyzed by the real VLM: {image_path}")
channel_checkpoint = PROJECT_DIR / report["checkpoints_dir"]
required_checkpoints = [channel_checkpoint / "inventory.json", channel_checkpoint / "sampling.json", channel_checkpoint / "agents.json", channel_checkpoint / "memory.json", channel_checkpoint / "dataset.json"]
required_checkpoints += list((channel_checkpoint / "videos").glob("*.json"))
required_checkpoints += list((channel_checkpoint / "agent_cycles").glob("cycle_*.json"))
if any(not path.is_file() for path in required_checkpoints):
    raise RuntimeError("BLOCKED: one or more inventory/analysis/agent/memory/dataset resume checkpoints are missing.")
training_status = report.get("training_status")
if training_status == "complete":
    training_checkpoints = PROJECT_DIR / "training/lora/adapter"
    if not list(training_checkpoints.glob("checkpoint-*")):
        raise RuntimeError("BLOCKED: LoRA completed without an epoch checkpoint.")
elif training_status not in ("skipped_quality_or_size_gate",):
    raise RuntimeError(f"BLOCKED: unexpected training status: {training_status}")
summary = {"status": "PASS", "videos": report["downloaded_and_analyzed_count"], "frames": report["extracted_frame_count"], "dataset_records": len(records), "checkpoint_files": len(required_checkpoints), "resume_enabled": True, "training_status": training_status, "mock_used": False}
print(json.dumps(summary, indent=2))